# Rough-Regime Volatility Arbitrage Engine
## Master Analysis Notebook — Full Computational Pipeline

**Project:** *Rough Volatility Arbitrage under Markov Regime Switching: A Volterra Process Approach with HMM Risk Gating*  
**Authors:** Quantitative Research Team  

This notebook is the **canonical** production pipeline. It executes every analytical component in order and saves high-resolution PNG artefacts for direct inclusion in the LaTeX manuscript.

| § | Component | Output File |
|---|-----------|-------------|
| 1 | Fractional Brownian Motion & Rough Volatility | `rough_vol_paths.png` |
| 2 | Hybrid Scheme Monte Carlo Convergence | `mc_convergence.png` |
| 3 | G-HMM Regime Detection | `regime_map.png` |
| 4 | Strategy Performance & PnL | `equity_curve.png` |
| 5 | Transaction Cost Sensitivity | `transaction_cost_heatmap.png` |
| 6 | Heston Stochastic Volatility | `heston_surface.png` |
| 7 | SABR Model | `sabr_smile.png` |
| 8 | Local Volatility (Dupire) | `dupire_local_vol.png` |
| 9 | Hawkes Jump-Diffusion | `hawkes_jump_diffusion.png` |

---

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# § 0 — ENVIRONMENT SETUP
# ═══════════════════════════════════════════════════════════════════
import warnings
warnings.filterwarnings('ignore')

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import norm as sp_norm
from scipy.special import gamma as gamma_fn
from scipy.optimize import brentq
from numba import njit

# ── Global plot style ────────────────────────────────────────────────
PALETTE = {
    'calm':      '#2196F3',
    'turbulent': '#F44336',
    'naive':     '#90A4AE',
    'regime':    '#4CAF50',
    'accent':    '#FF9800',
    'dark_bg':   '#0f0f0f',
}

plt.rcParams.update({
    'figure.dpi':        150,
    'font.family':       'serif',
    'font.size':         11,
    'axes.titlesize':    12,
    'axes.labelsize':    11,
    'legend.fontsize':   9,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.30,
})

RNG = np.random.default_rng(42)
SAVE_DPI = 150

print('Environment ready. All outputs will be saved as 150-dpi PNGs.')

---
## § 1 — Fractional Brownian Motion & Rough Volatility

Fractional Brownian motion $B^H_t$ with Hurst exponent $H \in (0,1)$ is the unique centred Gaussian process satisfying:

$$\mathbb{E}\bigl[B^H_t\, B^H_s\bigr] = \tfrac{1}{2}\bigl(t^{2H} + s^{2H} - |t-s|^{2H}\bigr)$$

For $H < \tfrac{1}{2}$, successive increments are **negatively correlated** (anti-persistent / rough). The empirically measured Hurst exponent for S&P 500 realised variance is $H \approx 0.07$.

Paths are generated by the **Davies–Harte exact circulant embedding** method, which avoids the $O(N^2)$ cost of direct Cholesky simulation.

In [ ]:
def davies_harte_fbm(n: int, H: float, rng=None) -> np.ndarray:
    """
    Exact circulant-embedding simulation of fractional Brownian motion
    on [0,1] with n steps (Davies & Harte, 1987).
    """
    if rng is None:
        rng = np.random.default_rng()
    k   = np.arange(n + 1, dtype=np.float64)
    cov = 0.5 * (np.abs(k+1)**(2*H) - 2.0*np.abs(k)**(2*H) + np.abs(k-1)**(2*H))
    row = np.empty(2*n)
    row[:n+1] = cov
    row[n+1:] = cov[n-1:0:-1]
    ev  = np.real(np.fft.fft(row))
    ev  = np.maximum(ev, 0.0)
    sq  = np.sqrt(ev / (2*n))
    Z   = rng.standard_normal(2*n) + 1j * rng.standard_normal(2*n)
    Z[0] = np.sqrt(2) * rng.standard_normal()
    if n % 2 == 0:
        Z[n] = np.sqrt(2) * rng.standard_normal()
    fgn = np.real(np.fft.ifft(sq * Z)) * np.sqrt(2*n)
    fgn = fgn[:n] * (1.0/n)**H
    return np.concatenate([[0.0], np.cumsum(fgn)])


def volterra_variance_path(n: int, H: float, v0: float, nu: float,
                           rng=None) -> np.ndarray:
    """
    Simulate the spot variance v_t as a Volterra process:
        v_t = v_0 + (1/Γ(H+1/2)) ∫₀ᵗ (t-s)^{H-1/2} ν√(v_s) dW_s
    using a first-order near-field kernel approximation.
    """
    if rng is None:
        rng = np.random.default_rng()
    dt    = 1.0 / n
    beta  = H + 0.5
    inv_g = 1.0 / gamma_fn(H + 0.5)
    kappa = min(12, n // 2)

    # Near-field kernel weights (exact power-law integration)
    w = np.array([
        (((j+1)*dt)**beta - (j*dt)**beta) / beta * inv_g
        for j in range(kappa)
    ])

    v   = np.zeros(n+1)
    v[0] = v0
    buf = np.zeros(kappa)
    bidx = 0

    for i in range(n):
        dW  = rng.standard_normal() * math.sqrt(dt)
        vp  = max(v[i], 1e-10)
        sv  = nu * math.sqrt(vp)
        near = w[0] * sv * dW
        for lag in range(1, min(i+1, kappa)):
            near += w[lag] * buf[(bidx - lag) % kappa]
        v[i+1]  = max(v0 + near, 0.0)
        buf[bidx] = sv * dW
        bidx = (bidx + 1) % kappa

    return v


# ── Simulation parameters ─────────────────────────────────────────────
N_STEPS = 1000
N_PATHS = 7
T_AXIS  = np.linspace(0, 1, N_STEPS+1)
H_CONFIG = {'H=0.07 (Rough, empirical SPX)': 0.07,
             'H=0.50 (Standard BM)':          0.50}

fbm_paths = {}
vol_paths = {}

for label, H in H_CONFIG.items():
    fbm_paths[label] = [davies_harte_fbm(N_STEPS, H, rng=np.random.default_rng(i*77))
                        for i in range(N_PATHS)]
    vol_paths[label] = [volterra_variance_path(N_STEPS, H, v0=0.04, nu=0.3,
                                               rng=np.random.default_rng(i*77))
                        for i in range(N_PATHS)]

print('fBM and Volterra paths generated.')

In [ ]:
# ── Figure 1: rough_vol_paths.png ─────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
fig.suptitle('Figure 1 — Fractional Brownian Motion & Rough Volatility\n'
             'Rough ($H=0.07$) vs Standard Brownian ($H=0.50$)',
             fontsize=13, fontweight='bold')

gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.42, wspace=0.30)

cmaps = [plt.cm.Reds, plt.cm.Blues]
cols_per = [cm(np.linspace(0.4, 0.88, N_PATHS)) for cm in cmaps]

for col, (label, H) in enumerate(H_CONFIG.items()):
    colors = cols_per[col]

    # Row 0: fBM paths
    ax = fig.add_subplot(gs[0, col])
    for i, path in enumerate(fbm_paths[label]):
        ax.plot(T_AXIS, path, lw=0.75, alpha=0.80, color=colors[i])
    ax.set_title(f'fBM Paths  —  {label}', fontsize=10)
    ax.set_xlabel('Time $t$'); ax.set_ylabel('$B^H_t$')

    # Row 1: Rolling realised volatility of fBM increments
    ax2 = fig.add_subplot(gs[1, col])
    for i, path in enumerate(fbm_paths[label]):
        incr = np.diff(path)
        rv   = pd.Series(incr).rolling(50).std().values
        ax2.plot(T_AXIS[50:], rv[49:], lw=0.8, alpha=0.70, color=colors[i])
    ax2.set_title(f'fBM Realised Vol (50-step window)  —  {label}', fontsize=10)
    ax2.set_xlabel('Time $t$'); ax2.set_ylabel('Rolling Std')

    # Row 2: Volterra variance paths (√v_t = instantaneous vol)
    ax3 = fig.add_subplot(gs[2, col])
    for i, vpath in enumerate(vol_paths[label]):
        ax3.plot(T_AXIS, np.sqrt(vpath) * 100, lw=0.85, alpha=0.75, color=colors[i])
    ax3.axhline(20.0, ls='--', lw=1.2, color='navy', alpha=0.7, label='$\\sqrt{v_0}=20\\%$')
    ax3.set_title(f'Volterra Instantaneous Vol $\\sqrt{{v_t}}$  —  {label}', fontsize=10)
    ax3.set_xlabel('Time $t$'); ax3.set_ylabel('Ann. Vol (%)')
    ax3.legend(fontsize=8)

plt.savefig('rough_vol_paths.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → rough_vol_paths.png')

---
## § 2 — Hybrid Scheme Monte Carlo Convergence

The kernel singularity $K(t,s) = (t-s)^{H-1/2}/\Gamma(H+\frac{1}{2}) \to +\infty$ as $s\to t^-$ for $H < \tfrac{1}{2}$ renders standard Euler–Maruyama inconsistent: strong error is $O(n^H)$ instead of $O(n^{1/2})$. The **Hybrid Scheme** (Bennedsen et al., 2017) corrects this by exact integration over the near-field and an exponential-sum approximation in the far-field.

The canonical Monte Carlo convergence rate $\mathrm{SE}(N) \propto N^{-1/2}$ is verified by fitting:
$$\log \widehat{\mathrm{SE}}(N) = a + b\,\log N, \qquad b \approx -0.50$$

In [ ]:
@njit(cache=True, fastmath=True)
def _mc_straddle_hybrid(n_paths, n_steps, dt, v0, nu, H,
                        S0, r, rho, inv_g, kappa, K, seed):
    """
    Numba-JIT Hybrid Scheme Monte Carlo for an ATM straddle.
    Near-field: exact power-law weights over kappa lags.
    Returns (mean_price, std_payoff).
    """
    sqdt  = math.sqrt(dt)
    beta  = H + 0.5
    w = np.empty(kappa)
    for j in range(kappa):
        upper = ((j+1)*dt)**beta
        lower = (j*dt)**beta if j > 0 else 0.0
        w[j]  = (upper - lower) / beta * inv_g

    payoffs = np.empty(n_paths)
    for p in range(n_paths):
        np.random.seed(seed + p)
        v = v0; logS = math.log(S0)
        buf = np.zeros(kappa); bidx = 0
        for i in range(n_steps):
            z1 = np.random.standard_normal()
            z2 = np.random.standard_normal()
            dWs = sqdt * z1
            dWv = sqdt * (rho*z1 + math.sqrt(max(1.0-rho**2, 0.0))*z2)
            vp  = max(v, 1e-9)
            sv  = nu * math.sqrt(vp)
            near = w[0] * sv * dWv
            for lag in range(1, min(i+1, kappa)):
                near += w[lag] * buf[(bidx-lag) % kappa]
            v    = max(v0 + near, 0.0)
            logS += (r - 0.5*v)*dt + math.sqrt(max(v, 0.0))*dWs
            buf[bidx] = sv * dWv
            bidx = (bidx+1) % kappa
        payoffs[p] = abs(math.exp(logS) - K)

    disc = math.exp(-r * n_steps * dt)
    return disc * payoffs.mean(), disc * payoffs.std()


def bs_straddle(S, K, T, r, sigma):
    d1  = (np.log(S/K) + (r+0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2  = d1 - sigma*np.sqrt(T)
    return (S*sp_norm.cdf(d1) - K*np.exp(-r*T)*sp_norm.cdf(d2) +
            K*np.exp(-r*T)*sp_norm.cdf(-d2) - S*sp_norm.cdf(-d1))


# ── Parameters (H=0.5, ν→0 so BS formula is exact reference) ─────────
MC_PARAMS = dict(S0=100.0, K=100.0, r=0.05, v0=0.04,
                 H=0.50, nu=1e-6, rho=0.0,
                 T_days=5, kappa=8, steps_per_day=16)

T_ann  = MC_PARAMS['T_days'] / 252.0
STEPS  = MC_PARAMS['T_days'] * MC_PARAMS['steps_per_day']
DT     = T_ann / STEPS
INV_G  = 1.0 / gamma_fn(MC_PARAMS['H'] + 0.5)

BS_REF = bs_straddle(MC_PARAMS['S0'], MC_PARAMS['K'],
                     T_ann, MC_PARAMS['r'], np.sqrt(MC_PARAMS['v0']))

# JIT warm-up
_mc_straddle_hybrid(10, STEPS, DT, MC_PARAMS['v0'], MC_PARAMS['nu'],
                    MC_PARAMS['H'], MC_PARAMS['S0'], MC_PARAMS['r'],
                    MC_PARAMS['rho'], INV_G, MC_PARAMS['kappa'],
                    MC_PARAMS['K'], 0)

print(f'BS straddle reference: ${BS_REF:.5f}')

# ── Convergence study ─────────────────────────────────────────────────
N_TRIALS = 25
N_LIST   = [32, 64, 128, 256, 512, 1024, 2048, 4096]
SE_list, ERR_list = [], []

for N in N_LIST:
    prices = [
        _mc_straddle_hybrid(N, STEPS, DT, MC_PARAMS['v0'], MC_PARAMS['nu'],
                            MC_PARAMS['H'], MC_PARAMS['S0'], MC_PARAMS['r'],
                            MC_PARAMS['rho'], INV_G, MC_PARAMS['kappa'],
                            MC_PARAMS['K'], seed=t*1000+N)[0]
        for t in range(N_TRIALS)
    ]
    arr = np.array(prices)
    SE_list.append(arr.std(ddof=1))
    ERR_list.append(abs(arr.mean() - BS_REF))

# Log-log regression
slope, intercept, r_val, *_ = stats.linregress(
    np.log(N_LIST), np.log(np.array(SE_list)+1e-12))
print(f'\nConvergence slope b = {slope:+.4f}  (expected -0.50)  R²={r_val**2:.4f}')

In [ ]:
# ── Figure 2: mc_convergence.png ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(r'Figure 2 — Monte Carlo Convergence: $\mathrm{SE}(N) \propto N^{-1/2}$',
             fontsize=13, fontweight='bold')

N_arr  = np.array(N_LIST)
SE_arr = np.array(SE_list)

# Left panel: log-log SE
ax = axes[0]
ax.scatter(N_arr, SE_arr, s=70, zorder=5,
           color=PALETTE['accent'], edgecolors='k', lw=0.6)
ax.plot(N_arr, SE_arr, color=PALETTE['accent'], lw=1.0, alpha=0.5)
fitted = np.exp(intercept + slope*np.log(N_arr))
ax.plot(N_arr, fitted, 'r--', lw=2.0,
        label=fr'Fitted: $b={slope:+.3f}$  $R^2={r_val**2:.3f}$')
ref_se = SE_arr[0] * np.sqrt(N_arr[0]/N_arr)
ax.plot(N_arr, ref_se, 'g:', lw=1.5, label=r'Theory $O(N^{-1/2})$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Number of Paths $N$')
ax.set_ylabel('Standard Error SE')
ax.set_title('Log–Log Standard Error vs $N$')
ax.legend(fontsize=9)

# Right panel: absolute error vs BS
ax2 = axes[1]
ax2.scatter(N_arr, ERR_list, s=70, zorder=5,
            color=PALETTE['calm'], edgecolors='k', lw=0.6)
ax2.plot(N_arr, ERR_list, color=PALETTE['calm'], lw=1.0, alpha=0.5)
err_ref = ERR_list[0] * np.sqrt(N_arr[0]/N_arr)
ax2.plot(N_arr, err_ref, 'g:', lw=1.5, label=r'$O(N^{-1/2})$ reference')
ax2.axhline(BS_REF*0.005, ls=':', lw=1, color='grey', alpha=0.7,
            label='0.5% of BS price')
ax2.set_xscale('log'); ax2.set_yscale('log')
ax2.set_xlabel('Number of Paths $N$')
ax2.set_ylabel(r'$|\hat{P}(N) - P_{\mathrm{BS}}|$  (\$)')
ax2.set_title('Absolute Error vs Black–Scholes Reference')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('mc_convergence.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → mc_convergence.png')

---
## § 3 — G-HMM Regime Detection

The Gaussian Hidden Markov Model specifies emission distributions:

$$r_t \mid s_t = k \;\sim\; \mathcal{N}(\mu_k,\, \sigma_k^2), \qquad k \in \{0 \text{ (Calm)},\; 1 \text{ (Turbulent)}\}$$

Parameters $\theta = (\boldsymbol{\pi}, A, \boldsymbol{\mu}, \boldsymbol{\sigma})$ are fitted via **Baum–Welch EM** with $K$-means++ initialisation and $R=8$ random restarts. The **traffic-light** rule gates positions:

- $P > 0.80$: **HALT** — flatten immediately
- $0.60 < P \leq 0.80$: **HEDGE** — reduce to delta-neutral
- $P \leq 0.60$: **TRADE** — full Kelly allocation

In [ ]:
class GaussianHMM2State:
    """
    Two-state Gaussian HMM — fully vectorised Baum-Welch.
    Implements forward, backward, Viterbi, and online update.
    """

    def __init__(self, n_iter=300, tol=1e-8):
        self.n_iter, self.tol = n_iter, tol

    def _log_b(self, obs, mu, sigma):
        return sp_norm.logpdf(obs[:, None], mu[None, :], sigma[None, :])

    def _forward(self, log_B, pi, A):
        T, K = log_B.shape
        alpha, lsc = np.zeros((T, K)), np.zeros(T)
        a0 = pi * np.exp(log_B[0]); s0 = a0.sum()
        lsc[0] = np.log(s0+1e-300); alpha[0] = a0/(s0+1e-300)
        for t in range(1, T):
            a = (alpha[t-1] @ A) * np.exp(log_B[t])
            s = a.sum(); lsc[t] = np.log(s+1e-300); alpha[t] = a/(s+1e-300)
        return alpha, lsc

    def _backward(self, log_B, A, lsc):
        T, K = log_B.shape
        beta = np.zeros((T, K)); beta[-1] = 1.0
        for t in range(T-2, -1, -1):
            b = A @ (np.exp(log_B[t+1]) * beta[t+1])
            beta[t] = b / (np.exp(lsc[t+1])+1e-300)
        return beta

    def fit(self, obs, n_restarts=8, seed=0):
        obs = np.asarray(obs, dtype=np.float64)
        rng = np.random.default_rng(seed)
        best = (-np.inf, None, None, None, None, None, None, [])

        for _ in range(n_restarts):
            # K-means++ initialisation
            i0  = rng.integers(len(obs))
            d2  = (obs - obs[i0])**2
            i1  = rng.choice(len(obs), p=d2/d2.sum())
            mu  = np.array([obs[i0], obs[i1]])
            sig = np.full(2, obs.std()) * rng.uniform(0.6, 1.4, 2)
            sig = np.maximum(sig, 1e-5)
            d   = rng.uniform(0.88, 0.97)
            A   = np.array([[d, 1-d], [1-d, d]])
            pi  = rng.dirichlet([1.0, 1.0])
            ll_prev = -np.inf
            ll_hist = []

            for _ in range(self.n_iter):
                log_B        = self._log_b(obs, mu, sig)
                alpha, lsc   = self._forward(log_B, pi, A)
                beta         = self._backward(log_B, A, lsc)
                ll           = lsc.sum()
                ll_hist.append(ll)
                if abs(ll - ll_prev) < self.tol:
                    break
                ll_prev = ll
                gamma = alpha * beta
                gamma /= gamma.sum(1, keepdims=True)+1e-300
                xi = np.einsum('tj,jk,tk,tk->jk',
                               alpha[:-1], A, np.exp(log_B[1:]), beta[1:])
                xi /= xi.sum()+1e-300
                pi = gamma[0]
                A  = xi / (xi.sum(1, keepdims=True)+1e-300)
                w  = gamma.sum(0)+1e-300
                mu = gamma.T @ obs / w
                sig = np.sqrt(((gamma * (obs[:,None]-mu)**2).sum(0)) / w)
                sig = np.maximum(sig, 1e-6)

            if ll > best[0]:
                best = (ll, pi, A, mu, sig, gamma, alpha, ll_hist)

        _, pi, A, mu, sig, gamma, alpha, ll_hist = best
        # Re-label: state 1 = turbulent (higher σ)
        if sig[0] > sig[1]:
            for arr in [mu, sig, pi]:
                arr[:] = arr[::-1]
            A     = A[::-1, ::-1]
            gamma = gamma[:, ::-1]
            alpha = alpha[:, ::-1]

        self.pi_ = pi; self.A_ = A; self.mu_ = mu; self.sigma_ = sig
        self.gamma_ = gamma; self.alpha_ = alpha
        self._ll_hist = ll_hist
        return self

    def viterbi(self, obs):
        log_B = self._log_b(obs, self.mu_, self.sigma_)
        logA  = np.log(self.A_+1e-300)
        T, K  = log_B.shape
        ld = np.log(self.pi_+1e-300) + log_B[0]
        psi = np.zeros((T, K), int)
        for t in range(1, T):
            cand = ld[:, None] + logA
            psi[t] = cand.argmax(0)
            ld = cand[psi[t], np.arange(K)] + log_B[t]
        q = np.empty(T, int); q[-1] = ld.argmax()
        for t in range(T-2, -1, -1):
            q[t] = psi[t+1, q[t+1]]
        return q

    def prob_turbulent(self, obs):
        """Forward-filtered P(turbulent | r_{1:t}) — no look-ahead bias."""
        log_B = self._log_b(obs, self.mu_, self.sigma_)
        alpha, _ = self._forward(log_B, self.pi_, self.A_)
        return alpha[:, 1]


# ── Synthetic data: 5 years daily with 5 turbulent episodes ─────────
np.random.seed(42)
T_SIM   = 1260
RETS    = np.random.normal(5e-4, 7e-3, T_SIM)
EVENTS  = [(200,260),(420,470),(600,660),(850,930),(1050,1110)]
for s, e in EVENTS:
    RETS[s:e] = np.random.normal(-1.5e-3, 2.5e-2, e-s)

TRUE_STATES = np.zeros(T_SIM, dtype=int)
for s, e in EVENTS:
    TRUE_STATES[s:e] = 1

# Fit HMM
hmm = GaussianHMM2State(n_iter=300)
hmm.fit(RETS, n_restarts=8, seed=0)

STATES    = hmm.viterbi(RETS)
PROB_TURB = hmm.prob_turbulent(RETS)

accuracy = (STATES == TRUE_STATES).mean()
dur_calm = 1.0 / (1.0 - hmm.A_[0, 0])
dur_turb = 1.0 / (1.0 - hmm.A_[1, 1])

print(f'Calm    μ={hmm.mu_[0]:+.6f}  σ={hmm.sigma_[0]:.6f}')
print(f'Turbulent μ={hmm.mu_[1]:+.6f}  σ={hmm.sigma_[1]:.6f}')
print(f'State accuracy: {accuracy:.1%}')
print(f'Expected durations: Calm={dur_calm:.0f}d  Turbulent={dur_turb:.0f}d')

In [ ]:
# ── Figure 3: regime_map.png ──────────────────────────────────────────
DATES = pd.bdate_range('2019-01-02', periods=T_SIM)
RV21  = pd.Series(RETS).rolling(21).std().values * np.sqrt(252)

fig = plt.figure(figsize=(16, 10))
fig.suptitle('Figure 3 — G-HMM Regime Map: Calm vs Turbulent State Separation',
             fontsize=13, fontweight='bold')
gs = gridspec.GridSpec(3, 2, figure=fig,
                        height_ratios=[3, 1.5, 1.5],
                        width_ratios=[3, 1],
                        hspace=0.08, wspace=0.28)

# Panel 1: returns scatter
ax1 = fig.add_subplot(gs[0, 0])
for s, col, lbl in [(0, PALETTE['calm'], 'Calm'),
                     (1, PALETTE['turbulent'], 'Turbulent')]:
    mask = STATES == s
    ax1.scatter(DATES[mask], RETS[mask], c=col, s=6, alpha=0.65,
                label=lbl, edgecolors='none')
for s, e in EVENTS:
    ax1.axvspan(DATES[s], DATES[e-1], color=PALETTE['turbulent'], alpha=0.07)
ax1.axhline(0, color='grey', lw=0.5, ls='--')
ax1.set_ylabel('Daily Log Return')
ax1.set_title(f'Returns by Viterbi State  |  Accuracy = {accuracy:.1%}')
ax1.legend(loc='upper left', fontsize=9)
ax1.tick_params(labelbottom=False)

# Panel 2: P(turbulent)
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax2.fill_between(DATES, PROB_TURB, alpha=0.5, color=PALETTE['turbulent'],
                  label='P(Turbulent)')
ax2.axhline(0.6, color='k', ls='--', lw=1.2, label='Threshold=0.60')
ax2.axhline(0.8, color='darkred', ls=':', lw=1.0, alpha=0.7, label='Halt=0.80')
ax2.set_ylim(0, 1); ax2.set_ylabel('P(Turbulent)')
ax2.legend(fontsize=8); ax2.tick_params(labelbottom=False)

# Panel 3: rolling vol
ax3 = fig.add_subplot(gs[2, 0], sharex=ax1)
ax3.plot(DATES, RV21, color='#37474F', lw=1.0)
for s, e in EVENTS:
    ax3.axvspan(DATES[s], DATES[e-1], color=PALETTE['turbulent'], alpha=0.08)
ax3.set_ylabel('21-day Ann. Vol'); ax3.set_xlabel('Date')

# Panel 4: distributions
ax4 = fig.add_subplot(gs[0:2, 1])
bins = np.linspace(RETS.min(), RETS.max(), 55)
xg   = np.linspace(RETS.min()*1.3, RETS.max()*1.3, 300)
for s, col, lbl in [(0, PALETTE['calm'], 'Calm'),
                     (1, PALETTE['turbulent'], 'Turbulent')]:
    mask = STATES == s
    ax4.hist(RETS[mask], bins=bins, orientation='horizontal',
             alpha=0.45, color=col, density=True, label=lbl)
    ax4.plot(sp_norm.pdf(xg, hmm.mu_[s], hmm.sigma_[s]),
             xg, color=col, lw=1.8)
ax4.set_xlabel('Density'); ax4.legend(fontsize=8)
ax4.set_title('Return Distributions')

# Panel 5: transition matrix heatmap
ax5 = fig.add_subplot(gs[2, 1])
im = ax5.imshow(hmm.A_, cmap='RdBu_r', vmin=0, vmax=1)
for (i, j), v in np.ndenumerate(hmm.A_):
    ax5.text(j, i, f'{v:.4f}', ha='center', va='center',
             fontsize=9, fontweight='bold')
ax5.set_xticks([0,1]); ax5.set_yticks([0,1])
ax5.set_xticklabels(['→Calm','→Turb'], fontsize=8)
ax5.set_yticklabels(['Calm→','Turb→'], fontsize=8)
ax5.set_title('Transition Matrix', fontsize=9)
plt.colorbar(im, ax=ax5, fraction=0.046)

plt.savefig('regime_map.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → regime_map.png')

---
## § 4 — Strategy Performance & Net PnL

Two strategies are compared over the 5-year synthetic history:

- **Naive:** constant 30% allocation to the long-vol straddle
- **Regime-Adjusted:** Kelly-optimal sizing, gated to zero when $P(\text{Turbulent}) > 0.60$

The Kelly fraction is:
$$f^* = \frac{\hat{\mu}_{63} - r_f}{\hat{\sigma}_{63}^2}, \qquad f^*_{\text{cap}} = \min(f^*, 0.50)$$

The net daily PnL after transaction costs:
$$\Pi_t^{\text{net}} = f_t \cdot r_t^{\text{strad}} - (c_{\text{spread}} + c_{\text{slip}}) \cdot |\Delta f_t|$$

In [ ]:
# ── Straddle return model ─────────────────────────────────────────────
KAPPA_VOL   = 8.0      # sensitivity to vol spread
IV_BAR      = 0.014    # constant implied vol proxy (daily)
EPS_STD     = 0.012    # idiosyncratic noise
RF_DAILY    = 5e-4 / 252
TURB_THRESH = 0.60
HALF_KELLY  = 0.50

RV5       = pd.Series(np.abs(RETS)).rolling(5, min_periods=1).mean().values
NOISE     = np.random.default_rng(7).normal(0, EPS_STD, T_SIM)
R_STRAD   = KAPPA_VOL * (RV5 - IV_BAR) + NOISE

# ── Kelly sizing ──────────────────────────────────────────────────────
s = pd.Series(R_STRAD)
mu_roll  = s.rolling(63, min_periods=21).mean().values
var_roll = s.rolling(63, min_periods=21).var().values
var_roll = np.where(var_roll < 1e-8, 1e-8, var_roll)
kelly    = np.clip((mu_roll - RF_DAILY) / var_roll, 0.0, HALF_KELLY)
kelly    = np.nan_to_num(kelly, nan=0.0)

# ── Positions ─────────────────────────────────────────────────────────
POS_NAIVE  = np.full(T_SIM, 0.30)
POS_REGIME = kelly * np.where(PROB_TURB > TURB_THRESH,
                              np.maximum(1.0 - 2.0*PROB_TURB, 0.0),
                              1.0)
POS_REGIME = np.clip(POS_REGIME, 0.0, HALF_KELLY)

# ── PnL & equity ─────────────────────────────────────────────────────
PNL_NAIVE  = POS_NAIVE  * R_STRAD
PNL_REGIME = POS_REGIME * R_STRAD
EQ_NAIVE   = (1 + PNL_NAIVE).cumprod()
EQ_REGIME  = (1 + PNL_REGIME).cumprod()

def drawdown_series(eq):
    hmax = pd.Series(eq).cummax()
    return ((eq - hmax) / hmax).values

DD_NAIVE  = drawdown_series(EQ_NAIVE)
DD_REGIME = drawdown_series(EQ_REGIME)

def annualised_sharpe(pnl, rf=RF_DAILY):
    exc = pnl - rf
    return exc.mean() / exc.std() * np.sqrt(252)

SR_N  = annualised_sharpe(PNL_NAIVE)
SR_R  = annualised_sharpe(PNL_REGIME)
MDD_N = DD_NAIVE.min()
MDD_R = DD_REGIME.min()

print(f'Naive:         Sharpe={SR_N:.3f}   MDD={MDD_N:.2%}   FinalEq={EQ_NAIVE[-1]:.4f}')
print(f'Regime-Adj:   Sharpe={SR_R:.3f}   MDD={MDD_R:.2%}   FinalEq={EQ_REGIME[-1]:.4f}')
print(f'Improvement:  ΔSharpe={SR_R-SR_N:+.3f}  ΔMDD={MDD_R-MDD_N:+.2%}')

In [ ]:
# ── Figure 4: equity_curve.png ────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
fig.suptitle('Figure 4 — Strategy Performance: Naive vs Regime-Adjusted\n'
             f'Sharpe: Naive={SR_N:.2f}  Regime={SR_R:.2f}  |  '
             f'MDD: Naive={MDD_N:.1%}  Regime={MDD_R:.1%}',
             fontsize=13, fontweight='bold')

def shade_crises(ax):
    for k, (s, e) in enumerate(EVENTS):
        ax.axvspan(DATES[s], DATES[e-1], color=PALETTE['turbulent'],
                   alpha=0.08, label='Crisis episode' if k == 0 else '')

# Equity curves
ax = axes[0]
ax.plot(DATES, EQ_NAIVE,  color=PALETTE['naive'],  lw=1.5,
        label=f'Naive (Sharpe={SR_N:.2f})')
ax.plot(DATES, EQ_REGIME, color=PALETTE['regime'], lw=2.0,
        label=f'Regime-Adjusted (Sharpe={SR_R:.2f})')
shade_crises(ax)
ax.set_ylabel('Portfolio Value')
ax.set_title('Cumulative Equity Curves')
ax.legend(fontsize=9)

# Drawdown
ax2 = axes[1]
ax2.fill_between(DATES, DD_NAIVE,  0, alpha=0.55,
                 color=PALETTE['naive'],  label=f'Naive (MDD={MDD_N:.1%})')
ax2.fill_between(DATES, DD_REGIME, 0, alpha=0.60,
                 color=PALETTE['regime'], label=f'Regime (MDD={MDD_R:.1%})')
shade_crises(ax2)
ax2.set_ylabel('Drawdown')
ax2.set_title('Drawdown — HMM Filter Avoids Crisis-Period Peaks')
ax2.legend(fontsize=9)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# Kelly position sizes
ax3 = axes[2]
ax3.fill_between(DATES, POS_NAIVE,  alpha=0.35, color=PALETTE['naive'],
                 label='Naive (constant)')
ax3.fill_between(DATES, POS_REGIME, alpha=0.55, color=PALETTE['regime'],
                 label='Kelly × HMM gate')
shade_crises(ax3)
ax3.set_ylabel('Position Size $f_t$')
ax3.set_xlabel('Date')
ax3.set_title('Dynamic Kelly Sizing with HMM Traffic-Light')
ax3.legend(fontsize=9)

plt.tight_layout()
plt.savefig('equity_curve.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → equity_curve.png')

---
## § 5 — Transaction Cost Sensitivity

Net daily PnL: $\Pi_t^{\text{net}} = f_t r_t^{\text{strad}} - (c_{\text{spread}} + c_{\text{slip}})|\Delta f_t|$

The Sharpe ratio sensitivity heatmap over the grid $c_{\text{spread}} \in [0,2\%]$, $c_{\text{slip}} \in [0,1\%]$ reveals the **breakeven cost frontier** (black contour where SR = 0).

In [ ]:
def sharpe_with_costs(pos, ret_s, c_spread, c_slip, rf=RF_DAILY):
    c_total   = c_spread + c_slip
    delta_pos = np.abs(np.diff(np.concatenate([[0], pos])))
    net_pnl   = pos * ret_s - c_total * delta_pos
    exc       = net_pnl - rf
    return exc.mean() / exc.std() * np.sqrt(252) if exc.std() > 0 else 0.0


SPREAD_G  = np.linspace(0.00, 0.020, 25)
SLIP_G    = np.linspace(0.00, 0.010, 25)
SR_NAIVE  = np.zeros((len(SLIP_G), len(SPREAD_G)))
SR_REGIME = np.zeros((len(SLIP_G), len(SPREAD_G)))

for i, slip in enumerate(SLIP_G):
    for j, spr in enumerate(SPREAD_G):
        SR_NAIVE[i, j]  = sharpe_with_costs(POS_NAIVE,  R_STRAD, spr, slip)
        SR_REGIME[i, j] = sharpe_with_costs(POS_REGIME, R_STRAD, spr, slip)

# ── Figure 5: transaction_cost_heatmap.png ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Figure 5 — Sharpe Ratio vs Transaction Costs  '
             '(black contour = SR=0 breakeven)',
             fontsize=13, fontweight='bold')

xl = [f'{v*100:.1f}' for v in SPREAD_G]
yl = [f'{v*100:.1f}' for v in SLIP_G]
tk_x = np.arange(0, len(SPREAD_G), 5)
tk_y = np.arange(0, len(SLIP_G),   5)

for ax, data, title in zip(
    axes,
    [SR_NAIVE, SR_REGIME, SR_REGIME - SR_NAIVE],
    ['Naive Strategy', 'Regime-Adjusted', 'Improvement (Regime − Naive)'],
):
    cmap_kw = dict(cmap='PuOr', vmin=-1.5, vmax=1.5) if 'Improvement' in title \
              else dict(cmap='RdYlGn', vmin=-1.0, vmax=3.0)
    sns.heatmap(data, ax=ax, **cmap_kw, cbar=True,
                xticklabels=False, yticklabels=False, linewidths=0)
    ax.set_xticks(tk_x); ax.set_xticklabels([xl[k] for k in tk_x], fontsize=7)
    ax.set_yticks(tk_y); ax.set_yticklabels([yl[k] for k in tk_y], fontsize=7)
    ax.set_xlabel('Bid-Ask Spread (%)'); ax.set_ylabel('Slippage (%)')
    ax.set_title(title)
    ax.contour(data, levels=[0], colors=['black'], linewidths=1.5)

plt.tight_layout()
plt.savefig('transaction_cost_heatmap.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → transaction_cost_heatmap.png')

---
## § 6 — Heston Stochastic Volatility Model

$$dS_t = r S_t \, dt + \sqrt{V_t} S_t \, dW_t^S$$
$$dV_t = \kappa(\theta - V_t)\,dt + \xi\sqrt{V_t}\,dW_t^V, \qquad d\langle W^S, W^V\rangle_t = \rho\,dt$$

Feller condition: $2\kappa\theta > \xi^2$ ensures $V_t > 0$ a.s. The **leverage effect** arises from $\rho < 0$: spot down, vol up, generating the observed downward IV skew.

In [ ]:
def heston_milstein(S0, V0, kappa, theta, xi, rho, r,
                    T=1.0, n_steps=252, n_paths=4000, seed=42):
    """Full-truncation Milstein scheme for Heston (S, V) paths."""
    rng = np.random.default_rng(seed)
    dt  = T / n_steps; sdt = np.sqrt(dt)
    s1mrho = np.sqrt(max(1 - rho**2, 0.0))

    S = np.full(n_paths, S0, dtype=np.float64)
    V = np.full(n_paths, V0, dtype=np.float64)
    S_T = np.empty(n_paths)

    for _ in range(n_steps):
        Z1 = rng.standard_normal(n_paths)
        Z2 = rng.standard_normal(n_paths)
        dWS = sdt * Z1
        dWV = sdt * (rho*Z1 + s1mrho*Z2)
        sqV = np.sqrt(np.maximum(V, 0.0))
        V   = np.maximum(
            V + kappa*(theta-V)*dt + xi*sqV*dWV + 0.25*xi**2*(dWV**2 - dt), 0.0)
        S   = S * np.exp((r - 0.5*V)*dt + sqV*dWS)

    return S


def bs_call(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        return max(S - K, 0.0)
    d1 = (np.log(S/K) + (r+0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*sp_norm.cdf(d1) - K*np.exp(-r*T)*sp_norm.cdf(d2)


def implied_vol_call(price, S, K, T, r):
    intrinsic = max(S - K, 0.0)
    if price <= intrinsic + 1e-10:
        return np.nan
    try:
        return brentq(lambda s: bs_call(S, K, T, r, s) - price, 1e-5, 5.0, xtol=1e-8)
    except Exception:
        return np.nan


# Parameters (Feller satisfied: 2·2·0.04=0.16 > 0.3²=0.09)
HP = dict(S0=100., V0=0.04, kappa=2.0, theta=0.04, xi=0.3, rho=-0.7, r=0.05)
print(f'Feller: 2κθ={2*HP["kappa"]*HP["theta"]:.3f} > ξ²={HP["xi"]**2:.3f}  → OK')

# Build IV surface
KS = np.array([0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20])
TS = np.array([0.083, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0])
H_SURF = np.full((len(TS), len(KS)), np.nan)

print('Computing Heston IV surface...')
for i, T in enumerate(TS):
    S_T_mc = heston_milstein(**HP, T=T, n_steps=max(int(T*252),40), n_paths=5000, seed=i)
    disc   = np.exp(-HP['r']*T)
    for j, k in enumerate(KS):
        K = HP['S0'] * k
        price = disc * np.maximum(S_T_mc - K, 0.0).mean()
        H_SURF[i, j] = implied_vol_call(price, HP['S0'], K, T, HP['r'])
print('Done.')

In [ ]:
# ── Figure 6: heston_surface.png ─────────────────────────────────────
fig = plt.figure(figsize=(14, 5))
fig.suptitle('Figure 6 — Heston Stochastic Volatility: IV Surface & Variance Paths',
             fontsize=13, fontweight='bold')
gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1.5, 1])

# 3D surface
ax1 = fig.add_subplot(gs[0], projection='3d')
KG, TG = np.meshgrid(KS, TS)
ax1.plot_surface(KG, TG, H_SURF*100, cmap='RdYlGn_r', edgecolor='none', alpha=0.88)
ax1.set_xlabel('Moneyness $K/S_0$', labelpad=7)
ax1.set_ylabel('Maturity $T$ (yr)', labelpad=7)
ax1.set_zlabel('Implied Vol (%)')
ax1.set_title(r'Heston IV Surface ($\rho=-0.7$, leverage effect)', fontsize=10)
ax1.view_init(elev=28, azim=-55)

# Sample variance paths
ax2 = fig.add_subplot(gs[1])
rng_h = np.random.default_rng(99)
dt_h  = 1/252; T_h = 1.0; n_h = 252
t_ax  = np.linspace(0, 1, n_h+1)
for k in range(60):
    V = HP['V0']
    vp = [V]
    for _ in range(n_h):
        dW = rng_h.standard_normal()
        V  = max(V + HP['kappa']*(HP['theta']-V)*dt_h +
                 HP['xi']*math.sqrt(max(V,0))*math.sqrt(dt_h)*dW, 0.0)
        vp.append(V)
    ax2.plot(t_ax, np.sqrt(vp)*100, lw=0.6,
             color=cm.plasma(0.15 + 0.70*k/60), alpha=0.5)
ax2.axhline(np.sqrt(HP['theta'])*100, ls='--', lw=1.8,
            color='navy', label=r'$\sqrt{\theta}$ long-run vol')
ax2.set_xlabel('Time (years)'); ax2.set_ylabel('Inst. Vol (%)')
ax2.set_title('Variance Process Paths (60 samples)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('heston_surface.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → heston_surface.png')

---
## § 7 — SABR Model: Implied Volatility Smile

$$dF_t = \sigma_t F_t^\beta \, dW_t^1, \qquad d\sigma_t = \alpha\,\sigma_t\,dW_t^2, \qquad d\langle W^1, W^2\rangle_t = \rho\,dt$$

The Hagan et al. (2002) analytic IV approximation is the de facto standard in IR/FX derivatives desks.

In [ ]:
def sabr_iv(F, K, T, alpha, beta, rho, nu):
    """Hagan et al. (2002) SABR implied Black vol approximation."""
    K = np.atleast_1d(np.asarray(K, dtype=float))
    iv = np.empty(len(K))
    for idx, k in enumerate(K):
        if abs(F - k) < 1e-9:          # ATM limit
            FK = F**(1-beta)
            B  = 1 + ((1-beta)**2/24*alpha**2/F**(2*(1-beta))
                      + rho*beta*nu*alpha/(4*FK)
                      + (2-3*rho**2)/24*nu**2) * T
            iv[idx] = alpha / FK * B
        else:
            FK_mid = np.sqrt(F*k)
            logFK  = np.log(F/k)
            ob     = 1 - beta
            z      = nu/alpha * FK_mid**ob * logFK
            chi_z  = np.log((np.sqrt(1-2*rho*z+z**2)+z-rho)/(1-rho))
            d1     = FK_mid**ob * (1 + ob**2/24*logFK**2 + ob**4/1920*logFK**4)
            B      = (1 + (ob**2/24*alpha**2/FK_mid**(2*ob)
                           + rho*beta*nu*alpha/(4*FK_mid**ob)
                           + (2-3*rho**2)/24*nu**2) * T)
            iv[idx] = alpha / d1 * (z/chi_z) * B
    return iv.squeeze()


F_IR = 0.03          # 3% forward rate (swaption ATM)
T_SABR = 1.0
KK_IR  = np.linspace(0.005, 0.065, 100)

SABR_SCENARIOS = {
    r'Symmetric  ($\rho=0$)':      dict(alpha=0.025, beta=0.5, rho=0.00, nu=0.4),
    r'Skew ($\rho=-0.4$)':         dict(alpha=0.025, beta=0.5, rho=-0.4, nu=0.4),
    r'High VolVol ($\nu=0.8$)':    dict(alpha=0.025, beta=0.5, rho=-0.2, nu=0.8),
    r'Log-normal ($\beta=1$)':      dict(alpha=0.020, beta=1.0, rho=-0.3, nu=0.4),
}
for nm, p in SABR_SCENARIOS.items():
    print(f'{nm:35s}  ATM={sabr_iv(F_IR, F_IR, T_SABR, **p)*100:.3f}%')

In [ ]:
# ── Figure 7: sabr_smile.png ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Figure 7 — SABR: Implied Vol Smile/Skew at T=1yr, F=3%',
             fontsize=13, fontweight='bold')

colors = ['#1f77b4','#d62728','#2ca02c','#9467bd']

ax = axes[0]
for (nm, p), c in zip(SABR_SCENARIOS.items(), colors):
    iv = sabr_iv(F_IR, KK_IR, T_SABR, **p)*100
    ax.plot(KK_IR*100, iv, lw=2, color=c, label=nm)
ax.axvline(F_IR*100, ls=':', lw=1.2, color='grey', label='ATM forward')
ax.set_xlabel('Strike (%)'); ax.set_ylabel('Implied Vol (%)')
ax.set_title('IV Smile: SABR Parameter Sensitivity')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ATM vol term structure
ax2 = axes[1]
Ts_ts = np.array([0.25, 0.5, 1.0, 2.0, 5.0, 10.0])
base  = dict(alpha=0.025, beta=0.5, rho=-0.3, nu=0.4)
for nu_v, c in zip([0.2, 0.4, 0.8], ['#1f77b4','#2ca02c','#d62728']):
    p = dict(**base); p['nu'] = nu_v
    atms = [sabr_iv(F_IR, F_IR, t, **p)*100 for t in Ts_ts]
    ax2.plot(Ts_ts, atms, lw=2, marker='o', ms=5, color=c,
             label=fr'$\nu={nu_v}$')
ax2.set_xlabel('Maturity $T$ (yr)'); ax2.set_ylabel('ATM Implied Vol (%)')
ax2.set_title('ATM Vol Term Structure vs Vol-of-Vol')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('sabr_smile.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → sabr_smile.png')

---
## § 8 — Local Volatility: Dupire Equation

Dupire (1994): any arbitrage-free call price surface $C(T,K)$ is consistent with a unique deterministic local vol:

$$\sigma_{\mathrm{loc}}^2(T,K) = \frac{\partial_T C + rK\,\partial_K C}{\tfrac{1}{2}K^2\,\partial_{KK}C}$$

This provides a **perfect, model-free fit** to all observable vanilla prices, making it the canonical model for path-dependent exotics.

In [ ]:
def synth_iv_surface(S0, r, T_grid, K_grid):
    """SPX-like synthetic IV surface: ATM term structure + skew."""
    iv = np.zeros((len(T_grid), len(K_grid)))
    for i, T in enumerate(T_grid):
        atm   = 0.18 + 0.03*np.exp(-2*T)
        skew  = -0.12/np.sqrt(T)
        kurt  =  0.04/T
        for j, K in enumerate(K_grid):
            x = np.log(K/S0) / (atm*np.sqrt(T))
            iv[i, j] = max(atm + skew*x + kurt*x**2, 0.005)
    return iv


def dupire_lv(S0, r, T_grid, K_grid, iv_surf):
    """Numerically invert the Dupire equation via finite differences."""
    C = np.array([[bs_call(S0, K, T, r, iv_surf[i, j])
                   for j, K in enumerate(K_grid)]
                  for i, T in enumerate(T_grid)])
    n_T, n_K = len(T_grid), len(K_grid)
    lv = np.full((n_T, n_K), np.nan)

    for i in range(1, n_T-1):
        dT = T_grid[i+1] - T_grid[i-1]
        for j in range(1, n_K-1):
            K  = K_grid[j]
            dK = K_grid[j+1] - K_grid[j-1]
            dC_dT   = (C[i+1,j] - C[i-1,j]) / dT
            dC_dK   = (C[i,j+1] - C[i,j-1]) / dK
            d2C_dK2 = (C[i,j+1] - 2*C[i,j] + C[i,j-1]) / (0.5*dK)**2
            num = dC_dT + r*K*dC_dK
            den = 0.5*K**2*d2C_dK2
            lv[i, j] = np.sqrt(num/den) if den > 1e-10 and num > 0 else iv_surf[i,j]

    lv[0,:]  = lv[1,:];  lv[-1,:] = lv[-2,:]
    lv[:,0]  = lv[:,1];  lv[:,-1] = lv[:,-2]
    return lv


S0_D  = 100.0; r_D  = 0.05
T_D   = np.linspace(0.05, 2.0, 30)
K_D   = np.linspace(70, 135, 40)
IV_D  = synth_iv_surface(S0_D, r_D, T_D, K_D)
LV_D  = dupire_lv(S0_D, r_D, T_D, K_D, IV_D)

print(f'Market IV  : [{IV_D.min()*100:.1f}%, {IV_D.max()*100:.1f}%]')
print(f'Dupire LV  : [{np.nanmin(LV_D)*100:.1f}%, {np.nanmax(LV_D)*100:.1f}%]')

In [ ]:
# ── Figure 8: dupire_local_vol.png ────────────────────────────────────
fig = plt.figure(figsize=(14, 5))
fig.suptitle('Figure 8 — Dupire Local Volatility: Market IV vs Calibrated $\\sigma(T,K)$',
             fontsize=13, fontweight='bold')
gs = gridspec.GridSpec(1, 2, figure=fig)

TT, KK = np.meshgrid(T_D, K_D, indexing='ij')

ax1 = fig.add_subplot(gs[0], projection='3d')
ax1.plot_surface(KK, TT, IV_D*100, cmap='viridis', edgecolor='none', alpha=0.88)
ax1.set_xlabel('Strike $K$'); ax1.set_ylabel('Maturity $T$ (yr)')
ax1.set_zlabel('Implied Vol (%)')
ax1.set_title('Market Implied Vol Surface (synthetic SPX)', fontsize=10)
ax1.view_init(elev=30, azim=-50)

ax2 = fig.add_subplot(gs[1], projection='3d')
ax2.plot_surface(KK, TT, LV_D*100, cmap='plasma', edgecolor='none', alpha=0.88)
ax2.set_xlabel('Strike $K$'); ax2.set_ylabel('Maturity $T$ (yr)')
ax2.set_zlabel('Local Vol $\\sigma(T,K)$ (%)')
ax2.set_title('Dupire Local Volatility (calibrated, perfect fit)', fontsize=10)
ax2.view_init(elev=30, azim=-50)

plt.tight_layout()
plt.savefig('dupire_local_vol.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → dupire_local_vol.png')

---
## § 9 — Hawkes Jump-Diffusion: Self-Exciting Contagion

The Hawkes process intensity:
$$\lambda(t) = \mu + \sum_{t_i < t} \alpha_H e^{-\beta_H(t-t_i)}$$

Each event raises the intensity by $\alpha_H$, which decays at rate $\beta_H$ — capturing cascade and contagion dynamics during market crises. Stationarity requires $\alpha_H/\beta_H < 1$.

**HMM ↔ Hawkes two-tier architecture:**  
The G-HMM identifies the macro regime; within the turbulent state, the Hawkes kernel governs micro-scale jump clustering.

In [ ]:
def simulate_hawkes(mu, alpha_h, beta_h, T, seed=42):
    """Ogata thinning algorithm for a univariate Hawkes process."""
    rng = np.random.default_rng(seed)
    events = []; t = 0.0; lam_star = mu
    while t < T:
        u = rng.uniform()
        w = -np.log(u) / lam_star
        tc = t + w
        if tc > T:
            break
        lam_true = mu + sum(alpha_h*np.exp(-beta_h*(tc-s)) for s in events)
        if rng.uniform() <= lam_true / lam_star:
            events.append(tc)
            lam_star = lam_true + alpha_h
        else:
            lam_star = lam_true
        t = tc
    return np.array(events)


def hawkes_intensity(events, mu, alpha_h, beta_h, t_grid):
    lam = np.full_like(t_grid, mu)
    for ev in events:
        mask = t_grid > ev
        lam[mask] += alpha_h * np.exp(-beta_h*(t_grid[mask]-ev))
    return lam


def hawkes_jd_path(S0, mu_s, sigma_s, mu_J, sigma_J,
                   mu_H, alpha_h, beta_h, T=1.0, dt=1/252, seed=42):
    """Hawkes jump-diffusion price path."""
    rng = np.random.default_rng(seed)
    n   = int(T/dt)
    t   = np.linspace(0, T, n+1)
    events = simulate_hawkes(mu_H, alpha_h, beta_h, T, seed=seed)
    ev_steps = set(np.searchsorted(t, events, side='right') - 1)
    kappa_comp = np.exp(mu_J + 0.5*sigma_J**2) - 1
    lam_bar    = mu_H / (1 - alpha_h/beta_h)
    logS = math.log(S0)
    logS_path = [logS]
    for i in range(n):
        dW    = rng.standard_normal() * math.sqrt(dt)
        drift = (mu_s - 0.5*sigma_s**2 - lam_bar*kappa_comp)*dt
        jump  = rng.normal(mu_J, sigma_J) if i in ev_steps else 0.0
        logS += drift + sigma_s*dW + jump
        logS_path.append(logS)
    return t, np.exp(logS_path), events


HJD = dict(S0=100, mu_s=0.05, sigma_s=0.15,
           mu_J=-0.04, sigma_J=0.03,
           mu_H=3.0, alpha_h=0.6, beta_h=2.0, T=1.0, dt=1/252)

print(f'Stationarity: α/β = {HJD["alpha_h"]/HJD["beta_h"]:.3f} < 1 ✓')
paths_jd = [hawkes_jd_path(**HJD, seed=s) for s in range(6)]

In [ ]:
# ── Figure 9: hawkes_jump_diffusion.png ───────────────────────────────
fig = plt.figure(figsize=(15, 10))
fig.suptitle('Figure 9 — Hawkes Jump-Diffusion: Self-Exciting Contagion Dynamics',
             fontsize=13, fontweight='bold')
gs = gridspec.GridSpec(3, 2, hspace=0.50, wspace=0.35)

# ① Multi-path prices
ax1 = fig.add_subplot(gs[0, :])
for k, (t, S, ev) in enumerate(paths_jd):
    al = 0.75 if k == 0 else 0.35
    lw = 1.5  if k == 0 else 0.8
    c  = '#d62728' if k == 0 else cm.Blues(0.35 + 0.10*k)
    ax1.plot(t*252, S, lw=lw, alpha=al, color=c,
             label='Representative path' if k == 0 else None)
    if k == 0:
        for ev_t in ev:
            ax1.axvline(ev_t*252, ls=':', lw=0.5, color='orange', alpha=0.6)
ax1.set_ylabel('Price $S_t$')
ax1.set_title('Hawkes JD Paths  (orange = jump times, visibly clustered)')
ax1.legend(); ax1.grid(alpha=0.2)

# ② Conditional intensity λ(t)
t_ref, S_ref, ev_ref = paths_jd[0]
t_fine = np.linspace(0, 1, 2000)
lam    = hawkes_intensity(ev_ref, HJD['mu_H'], HJD['alpha_h'], HJD['beta_h'], t_fine)

ax2 = fig.add_subplot(gs[1, 0])
ax2.fill_between(t_fine*252, lam, alpha=0.35, color='#ff7f0e')
ax2.plot(t_fine*252, lam, lw=1.2, color='#d62728')
ax2.axhline(HJD['mu_H'], ls='--', lw=1, color='navy', label=r'Baseline $\mu$')
ax2.set_xlabel('Day'); ax2.set_ylabel(r'$\lambda(t)$')
ax2.set_title('Conditional Intensity (self-exciting spikes)'); ax2.legend()

# ③ Jump clustering
ax3 = fig.add_subplot(gs[1, 1])
for rk, (_, _, ev) in enumerate(paths_jd):
    ax3.scatter(ev*252, np.full_like(ev, rk), s=14,
                color=cm.plasma(0.15+0.14*rk), alpha=0.85)
ax3.set_xlabel('Day'); ax3.set_ylabel('Path')
ax3.set_title('Jump Event Clustering (contagion across paths)')

# ④ Excitation sensitivity
ax4 = fig.add_subplot(gs[2, 0])
for ah, c in zip([0.1, 0.4, 0.7, 0.9],
                  ['#1f77b4','#2ca02c','#ff7f0e','#d62728']):
    ev_s = simulate_hawkes(HJD['mu_H'], ah, HJD['beta_h'], 1.0, seed=0)
    lam_s = hawkes_intensity(ev_s, HJD['mu_H'], ah, HJD['beta_h'], t_fine)
    ax4.plot(t_fine*252, lam_s, lw=1.1, color=c,
             label=fr'$\alpha_H={ah}$  ({len(ev_s)} events)')
ax4.set_xlabel('Day'); ax4.set_ylabel(r'$\lambda(t)$')
ax4.set_title('Excitation Sensitivity'); ax4.legend(fontsize=8)

# ⑤ Fat-tail return distribution
ax5 = fig.add_subplot(gs[2, 1])
all_ret = [np.log(hawkes_jd_path(**HJD, seed=s)[1][-1]/HJD['S0'])
           for s in range(300)]
all_ret = np.array(all_ret)
ax5.hist(all_ret, bins=40, density=True, alpha=0.65,
         color='#d62728', label='Hawkes JD')
xr = np.linspace(all_ret.min(), all_ret.max(), 200)
ax5.plot(xr, sp_norm.pdf(xr, all_ret.mean(), all_ret.std()),
         lw=2, color='navy', label='Gaussian fit')
ax5.set_xlabel('Log-return'); ax5.set_ylabel('Density')
ax5.set_title('Fat-tail Returns via Jump Clustering'); ax5.legend()

plt.savefig('hawkes_jump_diffusion.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → hawkes_jump_diffusion.png')

---
## § 10 — Master Artefact Summary

| # | File | Section in Paper | Description |
|---|------|-----------------|-------------|
| 1 | `rough_vol_paths.png` | §IV.A Roughness Visualisation | fBM paths (H=0.07 vs H=0.50) + Volterra variance |
| 2 | `mc_convergence.png` | §IV.B MC Convergence | Log-log SE decay, slope ≈ −0.50 |
| 3 | `regime_map.png` | §IV.C Regime Detection | 5-panel HMM map, Viterbi + P(Turbulent) |
| 4 | `equity_curve.png` | §IV.D Strategy Performance | Equity, drawdown, Kelly sizing |
| 5 | `transaction_cost_heatmap.png` | §IV.E Transaction Costs | Sharpe heatmap + breakeven contour |
| 6 | `heston_surface.png` | §VI.A Heston | 3D IV surface + variance paths |
| 7 | `sabr_smile.png` | §VI.B SABR | Smile/skew + ATM term structure |
| 8 | `dupire_local_vol.png` | §VI.C Local Vol | Market IV vs calibrated Dupire surface |
| 9 | `hawkes_jump_diffusion.png` | §VI.D Hawkes | Self-exciting paths, intensity, clustering |

All outputs are 150 dpi PNGs saved to the working directory for direct `\includegraphics` inclusion in `main.tex`.